In [7]:
# Fix: common imports to ensure aliases are defined
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


# Optimización de hiperparámetros (GridSearchCV/RandomizedSearchCV), documentación de resultados.

**¿Qué son los hiperparámetros?**
Los hiperparámetros son configuraciones internas de los algoritmos que se fijan antes del entrenamiento.

No se aprenden de los datos, sino que determinan cómo el modelo construye sus reglas.

Ejemplo: en un Random Forest, decidir cuántos árboles usar o qué tan profundos pueden ser.

Inicialmente, los modelos se entrenaron con valores por defecto. En este paso se aplicó optimización de hiperparámetros para buscar configuraciones más adecuadas.

**¿Qué es GridSearchCV?**
Es una técnica que prueba todas las combinaciones posibles de hiperparámetros definidos en un “grid” (rejilla).

Usa validación cruzada (cv) para evaluar cada combinación en distintos subconjuntos de los datos.

Devuelve:

**best_params_:** la mejor combinación encontrada.

**best_score_:** el valor de la métrica con esa combinación.



## Optimización de hiperparámetros – Clasificación (Antecedentes personales de cáncer)

In [11]:
# Uso de utilidades de tuning desde src/

import sys
from pathlib import Path
import json
import pandas as pd

sys.path.append(str(Path.cwd().parent))

from src.data_preprocessing import (
    load_data,
    encode_categoricals
)

from src.hyperparameter_tuning import (
    grid_search_and_save,
    randomized_search_and_save
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler



repo_root = Path.cwd().resolve()

while not (repo_root / 'src').exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent



results_dir = repo_root / 'results'
metrics_dir = results_dir / 'metrics'

metrics_dir.mkdir(parents=True, exist_ok=True)



df = load_data("dataset_chile_cancer_piel.csv")

y = df['Antecedentes personales de cáncer'].map({'Sí': 1, 'No': 0})

X = df.drop([
    'Antecedentes personales de cáncer',
    'Cáncer familiar 1er grado (tipo)'
], axis=1)

X = encode_categoricals(X, drop_first=True)



X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



log_reg = LogisticRegression(max_iter=1000, random_state=42)

param_grid_log = {
    "C": [0.01, 0.1, 1, 10],
    "penalty": ["l2"],
    "solver": ["lbfgs"]
}

grid_log = grid_search_and_save(
    log_reg,
    param_grid_log,
    X_train_scaled,
    y_train,
    out_csv=metrics_dir / 'grid_logistic_cv_results.csv',
    cv=5,
    scoring='f1'
)

print("Mejores parámetros Logistic Regression:", grid_log.best_params_)
print("Mejor F1 validación:", grid_log.best_score_)

with open(metrics_dir / 'grid_logistic_best_params.json', 'w', encoding='utf-8') as f:
    json.dump(grid_log.best_params_, f)



rf_clf = RandomForestClassifier(random_state=42)

param_grid_rf = {
    "n_estimators": [50, 100, 200],
    "max_depth": [2, 4, 6, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

grid_rf = grid_search_and_save(
    rf_clf,
    param_grid_rf,
    X_train,
    y_train,
    out_csv=metrics_dir / 'grid_rf_cv_results.csv',
    cv=5,
    scoring='f1'
)

print("Mejores parámetros Random Forest:", grid_rf.best_params_)
print("Mejor F1 validación:", grid_rf.best_score_)

with open(metrics_dir / 'grid_rf_best_params.json', 'w', encoding='utf-8') as f:
    json.dump(grid_rf.best_params_, f)

Mejores parámetros Logistic Regression: {'C': 10, 'penalty': 'l2', 'solver': 'lbfgs'}
Mejor F1 validación: 0.1465268079062611
Mejores parámetros Random Forest: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 50}
Mejor F1 validación: 0.08051203277009729



Variable objetivo  
La tarea consiste en predecir la presencia de antecedentes personales de cáncer (Sí/No).

Logistic Regression
Mejores parámetros:

C = 10

penalty = l2

solver = lbfgs

Mejor F1 en validación cruzada ≈ 0.147

Interpretación:
El modelo logra un F1 bajo, lo que indica que su capacidad para equilibrar precisión y recall en la detección de casos positivos es limitada. Aunque el ajuste con C=10 reduce la regularización y permite mayor flexibilidad, el modelo no logra capturar patrones predictivos relevantes en los datos.

Random Forest Classifier
Mejores parámetros:

n_estimators = 50

max_depth = None (árboles sin límite de profundidad)

min_samples_split = 2

min_samples_leaf = 1

Mejor F1 en validación cruzada ≈ 0.081

Interpretación:
El modelo obtiene un F1 aún más bajo que Logistic Regression. Aunque los parámetros permiten árboles más complejos, el desempeño indica que el modelo tiende a favorecer la clase mayoritaria y no logra detectar adecuadamente los casos positivos.

Conclusión global
Ambos modelos presentan F1 muy bajos, lo que significa que no logran identificar de manera confiable los pacientes con antecedentes de cáncer.

Logistic Regression ofrece un desempeño ligeramente mejor que Random Forest, pero sigue siendo insuficiente.

El problema principal no está en los hiperparámetros, sino en el desbalance de clases y en la falta de variables predictivas fuertes en el dataset.

## Optimización de hiperparámetros – Regresión (Tamaño máximo de la lesión)

In [10]:
import pandas as pd
import json
from pathlib import Path

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from src.hyperparameter_tuning import grid_search_and_save



repo_root = Path.cwd().resolve()

while not (repo_root / 'src').exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent



results_dir = repo_root / 'results'
metrics_dir = results_dir / 'metrics'

metrics_dir.mkdir(parents=True, exist_ok=True)



y_reg = df["Tamaño máximo (cm)"]
X_reg = df.drop(["Tamaño máximo (cm)"], axis=1)

# Codificación de variables categóricas
X_reg = pd.get_dummies(X_reg, drop_first=True)



X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg,
    test_size=0.2,
    random_state=42
)



scaler_r = StandardScaler()

X_train_r_scaled = scaler_r.fit_transform(X_train_r)
X_test_r_scaled = scaler_r.transform(X_test_r)


lin_reg = LinearRegression()
lin_reg.fit(X_train_r_scaled, y_train_r)

y_pred_lin = lin_reg.predict(X_test_r_scaled)

print("--- Linear Regression ---")
print("MAE:", mean_absolute_error(y_test_r, y_pred_lin))
print("MSE:", mean_squared_error(y_test_r, y_pred_lin))
print("R²:", r2_score(y_test_r, y_pred_lin))


rf_reg = RandomForestRegressor(random_state=42)

param_grid_reg = {
    "n_estimators": [50, 100, 200],
    "max_depth": [2, 4, 6, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

grid_rf_reg = grid_search_and_save(
    rf_reg,
    param_grid_reg,
    X_train_r,
    y_train_r,
    out_csv=metrics_dir / 'grid_rf_reg_cv_results.csv',
    cv=5,
    scoring='r2'
)

print("Mejores parámetros Random Forest Regressor:", grid_rf_reg.best_params_)
print("Mejor R² validación:", grid_rf_reg.best_score_)



with open(metrics_dir / 'grid_rf_reg_best_params.json', 'w', encoding='utf-8') as f:
    json.dump(grid_rf_reg.best_params_, f)

--- Linear Regression ---
MAE: 1.5730349551594276
MSE: 3.3099087761447676
R²: -0.023554707459312096
Mejores parámetros Random Forest Regressor: {'max_depth': 2, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}
Mejor R² validación: -0.022852004178943775


**Linear Regression**
El modelo lineal no logró explicar la variabilidad del tamaño de la lesión.

El R² negativo indica que predice peor que usar simplemente la media.

Conclusión: la regresión lineal no es eficiente en este contexto.

**Random Forest Regressor**
Se probó con distintos hiperparámetros:
n_estimators: número de árboles (200).

max_depth: profundidad máxima (2).

min_samples_split: mínimo de muestras para dividir un nodo (2).

min_samples_leaf: mínimo de muestras en una hoja (2).

Aunque se encontró la mejor combinación, el R² siguió siendo negativo.

Conclusión: la regresión no mejora ni siquiera con optimización de parámetros